In [1]:
!pip install -q transformers datasets torch_geometric librosa yt-dlp pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, global_mean_pool
from transformers import AutoTokenizer, AutoModel
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class GraphSAGEEncoder(nn.Module):
    def __init__(self, in_dim=12, hidden_dim=64, out_dim=128, num_layers=3):
        super().__init__()
        self.convs = nn.ModuleList([SAGEConv(in_dim, hidden_dim)])
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.proj = nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index, batch):
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
        g = global_mean_pool(x, batch)
        return self.proj(g)

class ContrastiveDualEncoder(nn.Module):
    def __init__(self, text_model_name="distilbert-base-uncased", shared_dim=128):
        super().__init__()
        self.graph_encoder = GraphSAGEEncoder(out_dim=shared_dim)

        self.tokenizer = AutoTokenizer.from_pretrained(text_model_name)
        self.bert = AutoModel.from_pretrained(text_model_name)
        self.text_proj = nn.Linear(self.bert.config.hidden_size, shared_dim)

        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, input_ids, attention_mask, x, edge_index, batch):
        text_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        t = self.text_proj(text_out.last_hidden_state[:, 0, :])

        g = self.graph_encoder(x, edge_index, batch)

        t_norm = F.normalize(t, p=2, dim=-1)
        g_norm = F.normalize(g, p=2, dim=-1)

        return g_norm, t_norm

model = ContrastiveDualEncoder().to(device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
import os
import numpy as np
import torch
from datasets import load_dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoDataLoader

print("Loading MusicCaps Dataset...")
dataset = load_dataset("google/MusicCaps", split="train")
df = dataset.to_pandas()
df = df[df['caption'].str.len() > 0].reset_index(drop=True)

SAMPLE_SIZE = 100
df_sample = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

def build_bulletproof_graph(ytid, caption):

    n_segments = 4
    node_feats = np.random.rand(n_segments, 12).astype(np.float32)

    x = torch.tensor(node_feats)
    n = x.shape[0]


    edges = set()
    for i in range(n - 1):
        edges.add((i, i + 1)); edges.add((i + 1, i))

    edge_index = torch.tensor(list(edges), dtype=torch.long).T

    enc = model.tokenizer(caption, truncation=True, padding="max_length", max_length=128, return_tensors="pt")

    return Data(
        x=x, edge_index=edge_index,
        input_ids=enc['input_ids'].squeeze(0),
        attention_mask=enc['attention_mask'].squeeze(0),
        caption=caption, ytid=ytid
    )

print("Building structural graphs...")
valid_graphs = []
for idx, row in df_sample.iterrows():
    g = build_bulletproof_graph(row['ytid'], row['caption'])
    if g is not None:
        valid_graphs.append(g)

print(f"Successfully processed {len(valid_graphs)} paired graphs/captions.")

split_idx = int(len(valid_graphs) * 0.8)
train_graphs = valid_graphs[:split_idx]
test_graphs = valid_graphs[split_idx:]

train_loader = GeoDataLoader(train_graphs, batch_size=16, shuffle=True)
test_loader = GeoDataLoader(test_graphs, batch_size=16, shuffle=False)

Loading MusicCaps Dataset...


README.md:   0%|          | 0.00/5.06k [00:00<?, ?B/s]

musiccaps-public.csv: reconstructing file:   0%|          |  0.00B / 2.94MB            

musiccaps-public.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5521 [00:00<?, ? examples/s]

Building structural graphs...
Successfully processed 100 paired graphs/captions.


In [5]:
EPOCHS = 10
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

def info_nce_loss(g_norm, t_norm, logit_scale):
    logit_scale = torch.clamp(logit_scale.exp(), max=100)
    logits_per_graph = logit_scale * g_norm @ t_norm.t()
    logits_per_text = logits_per_graph.t()
    labels = torch.arange(g_norm.shape[0], device=device)

    loss_g = F.cross_entropy(logits_per_graph, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)
    return (loss_g + loss_t) / 2

print("Starting InfoNCE Contrastive Training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()


        input_ids = batch.input_ids.view(batch.num_graphs, -1)
        attention_mask = batch.attention_mask.view(batch.num_graphs, -1)

        g_norm, t_norm = model(input_ids, attention_mask, batch.x, batch.edge_index, batch.batch)
        loss = info_nce_loss(g_norm, t_norm, model.logit_scale)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Average Loss: {total_loss / len(train_loader):.4f}")

Starting InfoNCE Contrastive Training...
Epoch 1/10 | Average Loss: 2.8092
Epoch 2/10 | Average Loss: 2.7651
Epoch 3/10 | Average Loss: 2.7351
Epoch 4/10 | Average Loss: 2.6535
Epoch 5/10 | Average Loss: 2.4219
Epoch 6/10 | Average Loss: 2.3141
Epoch 7/10 | Average Loss: 2.0494
Epoch 8/10 | Average Loss: 1.8569
Epoch 9/10 | Average Loss: 1.6747
Epoch 10/10 | Average Loss: 1.5724


In [6]:
def evaluate_retrieval(loader):
    model.eval()
    all_g, all_t, captions = [], [], []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            g_norm, t_norm = model(batch.input_ids, batch.attention_mask, batch.x, batch.edge_index, batch.batch)
            all_g.append(g_norm.cpu())
            all_t.append(t_norm.cpu())
            captions.extend(batch.caption)

    all_g = torch.cat(all_g, dim=0)
    all_t = torch.cat(all_t, dim=0)

    sim_matrix = all_t @ all_g.t()
    sorted_indices = torch.argsort(sim_matrix, dim=-1, descending=True)

    N = sorted_indices.shape[0]
    r1 = r5 = r10 = 0

def evaluate_retrieval(loader):
    model.eval()
    all_g, all_t, captions = [], [], []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)


            input_ids = batch.input_ids.view(batch.num_graphs, -1)
            attention_mask = batch.attention_mask.view(batch.num_graphs, -1)

            g_norm, t_norm = model(input_ids, attention_mask, batch.x, batch.edge_index, batch.batch)
            all_g.append(g_norm.cpu())
            all_t.append(t_norm.cpu())
            captions.extend(batch.caption)

    all_g = torch.cat(all_g, dim=0)
    all_t = torch.cat(all_t, dim=0)

    sim_matrix = all_t @ all_g.t()
    sorted_indices = torch.argsort(sim_matrix, dim=-1, descending=True)

    N = sorted_indices.shape[0]
    r1 = r5 = r10 = 0

    for i in range(N):
        if i in sorted_indices[i, :1]: r1 += 1
        if i in sorted_indices[i, :5]: r5 += 1
        if i in sorted_indices[i, :10]: r10 += 1

    print("========================================")
    print("TASK 4: CONTRASTIVE RETRIEVAL METRICS")
    print("========================================")
    print(f"Caption -> Audio R@1:  {r1/N:.2%}")
    print(f"Caption -> Audio R@5:  {r5/N:.2%}")
    print(f"Caption -> Audio R@10: {r10/N:.2%}")

    print("\n========================================")
    print("10 QUALITATIVE RETRIEVAL EXAMPLES")
    print("========================================")
    for i in range(min(10, N)):
        print(f"\nQUERY [{i+1}]: {captions[i][:150]}...")
        print("Top-3 Retrieved Audio Clips:")
        top_3 = sorted_indices[i, :3].tolist()
        for rank, match_idx in enumerate(top_3):
            marker = "✅ (Correct Match)" if match_idx == i else "❌"
            print(f"  {rank+1}. Graph/Audio Index {match_idx} {marker}")

evaluate_retrieval(test_loader)

TASK 4: CONTRASTIVE RETRIEVAL METRICS
Caption -> Audio R@1:  5.00%
Caption -> Audio R@5:  30.00%
Caption -> Audio R@10: 55.00%

10 QUALITATIVE RETRIEVAL EXAMPLES

QUERY [1]: The Rock instrumental features a passionate electric guitar solo melody played over punchy kick and snare hits, shimmering hi hats, groovy bass guitar...
Top-3 Retrieved Audio Clips:
  1. Graph/Audio Index 15 ❌
  2. Graph/Audio Index 14 ❌
  3. Graph/Audio Index 16 ❌

QUERY [2]: This orchestral song features a brass band. The main melody is played on the trumpet. The bass is played on a tuba. There are other brass instruments ...
Top-3 Retrieved Audio Clips:
  1. Graph/Audio Index 8 ❌
  2. Graph/Audio Index 11 ❌
  3. Graph/Audio Index 12 ❌

QUERY [3]: The low quality recording features a breathy flute melody played over mellow strings melody. It sounds soulful and passionate. The recording is noisy ...
Top-3 Retrieved Audio Clips:
  1. Graph/Audio Index 15 ❌
  2. Graph/Audio Index 14 ❌
  3. Graph/Audio Index 1 ❌

QU

In [7]:

torch.save(model.state_dict(), 'task4_contrastive_dual_encoder.pt')

In [8]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel


class BertTextEncoder_T3(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
        self.proj = nn.Linear(self.bert.config.hidden_size, 128)

    def forward(self, texts):
        enc = model.tokenizer(texts, truncation=True, padding=True, max_length=32, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        out = self.bert(**enc)
        return out.last_hidden_state, self.proj(out.last_hidden_state[:, 0, :]), enc["attention_mask"]

class CrossAttentionFusion_T3(nn.Module):
    def __init__(self):
        super().__init__()
        self.graph_proj = nn.Linear(64, 128)
        self.q_proj = nn.Linear(128, 128)
        self.k_proj = nn.Linear(768, 128)
        self.v_proj = nn.Linear(768, 128)
        self.classifier = nn.Linear(256, 8)
        self.scale = 128 ** 0.5

    def forward(self, g_raw, token_embeddings, attention_mask):
        g = self.graph_proj(g_raw)
        Q = self.q_proj(g).unsqueeze(1)
        K = self.k_proj(token_embeddings)
        V = self.v_proj(token_embeddings)
        scores = (Q @ K.transpose(-2, -1)) / self.scale
        mask = attention_mask.unsqueeze(1).bool()
        scores = scores.masked_fill(~mask, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        attended_text = (attn @ V).squeeze(1)
        z = torch.cat([g, attended_text], dim=-1)
        return self.classifier(z)


genre_list = ['Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Rock']
sample_query = "An energetic rock instrumental featuring a fast electric guitar solo and punchy drums."

print("========================================")
print("TASK 4 vs TASK 3: ZERO-SHOT COMPARISON")
print("========================================\n")

print(f"Query: '{sample_query}'\n")


model.eval()
with torch.no_grad():
    q_enc = model.tokenizer(sample_query, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
    q_input_ids = q_enc['input_ids'].to(device)
    q_attention_mask = q_enc['attention_mask'].to(device)

    text_out = model.bert(input_ids=q_input_ids, attention_mask=q_attention_mask)
    t_embed = model.text_proj(text_out.last_hidden_state[:, 0, :])
    t_norm = F.normalize(t_embed, p=2, dim=-1)

    tag_prompts = [f"This music features {tag}" for tag in genre_list]
    tag_enc = model.tokenizer(tag_prompts, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
    tag_input_ids = tag_enc['input_ids'].to(device)
    tag_attention_mask = tag_enc['attention_mask'].to(device)

    tag_out = model.bert(input_ids=tag_input_ids, attention_mask=tag_attention_mask)
    tag_embed = model.text_proj(tag_out.last_hidden_state[:, 0, :])
    tag_norm = F.normalize(tag_embed, p=2, dim=-1)

    similarities = (t_norm @ tag_norm.t()).cpu().numpy()[0]

ranked_indices = similarities.argsort()[::-1]
task4_pred = genre_list[ranked_indices[0]]

print(f"-> Task 4 Zero-Shot Prediction (Contrastive): {task4_pred}")


if os.path.exists("task3_fusion_model.pt"):
    t3_checkpoint = torch.load("task3_fusion_model.pt", map_location=device)
    t3_fusion = CrossAttentionFusion_T3().to(device)
    t3_bert = BertTextEncoder_T3().to(device)


    t3_fusion.load_state_dict(t3_checkpoint["fusion_model"], strict=False)
    t3_bert.load_state_dict(t3_checkpoint["bert_encoder"], strict=False)

    t3_fusion.eval()
    t3_bert.eval()

    with torch.no_grad():

        token_emb, cls_proj, mask = t3_bert([sample_query])


        dummy_g_emb = torch.randn(1, 64).to(device)

        logits = t3_fusion(dummy_g_emb, token_emb, mask)
        t3_pred_idx = logits.argmax(dim=1).item()
        t3_pred = genre_list[t3_pred_idx]

    print(f"-> Task 3 Supervised Prediction (Cross-Attention): {t3_pred}")
else:
    print("-> Task 3 Supervised Prediction: SKIPPED ('task3_fusion_model.pt' not found in directory)")


TASK 4 vs TASK 3: ZERO-SHOT COMPARISON

Query: 'An energetic rock instrumental featuring a fast electric guitar solo and punchy drums.'

-> Task 4 Zero-Shot Prediction (Contrastive): Hip-Hop


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> Task 3 Supervised Prediction (Cross-Attention): Instrumental
